In [ ]:
# セル1：ライブラリのインポート
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict

print("ライブラリの読み込みが完了しました。")

In [ ]:
# セル2：メインフォルダの選択

# ダイアログ用ウィンドウを裏で作成（最前面に固定）
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

# フォルダ選択ダイアログを開く
main_dir = filedialog.askdirectory(title="解析対象のメインフォルダを選択してください")

if not main_dir:
    print("⚠️ フォルダ選択がキャンセルされました。次のセルには進まず、やり直してください。")
else:
    print(f"✅ 選択されたメインフォルダ:\n{main_dir}")

In [ ]:
# セル3：対象ファイルのリストアップ（事前確認）

if not main_dir:
    raise ValueError("メインフォルダが選択されていません。セル2を再実行してフォルダを選んでください。")

target_files = []

# os.walkでフォルダの最深部までCSVファイルを探す
for dirpath, dirnames, filenames in os.walk(main_dir):
    for f in filenames:
        if f.lower().endswith('.csv'):
            # ファイルのフルパスを作成してリストに追加
            target_files.append(os.path.join(dirpath, f))

print(f"🔍 見つかったCSVファイル: 合計 {len(target_files)} 件")

# 最初の数件だけ試しにパスを表示して確認
if target_files:
    print("\n【処理対象ファイルの例】")
    for f in target_files[:3]:
        print(" -", f)
    if len(target_files) > 3:
        print("   ... (他省略)")
else:
    print("⚠️ CSVファイルが見つかりませんでした。フォルダを確認してください。")

In [ ]:
# セル4：設定とファイル整理（準備）

if 'target_files' not in locals() or not target_files:
    raise ValueError("処理対象のCSVファイルがありません。セル3を再実行してファイルを確認してください。")

# ==========================================
# ⚙️ 設定：解析したい列のリスト（ここで増減可能）
target_columns = ['CH03', 'CH04']
# ==========================================

# --- 画像をまとめる出力フォルダの設定 ---
current_dir = os.getcwd()
output_dir_name = "fft_results_per_channel" 
output_dir_path = os.path.join(current_dir, output_dir_name)
os.makedirs(output_dir_path, exist_ok=True)

# 全ファイルを「末端フォルダ」ごとにグループ化
folder_to_files = defaultdict(list)
for file_path in target_files:
    dirpath = os.path.dirname(file_path)
    folder_to_files[dirpath].append(file_path)

dt = 0.001  # サンプリング間隔 (1ms)
# カラーパレットの設定（統合版グラフで列ごとに色を変えるため）
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

print(f"✅ 準備完了。")
print(f"📁 出力先フォルダ: {output_dir_path}")
print(f"🎯 処理対象の列: {', '.join(target_columns)}")
print(f"📦 処理対象の末端フォルダ数: {len(folder_to_files)} 件\n")
print("👉 次の「セル5」を実行してください。")

In [ ]:
# セル5：列ごとの重ね描きグラフ作成（個別版）

print("🚀 【個別版】列ごとの重ね描きグラフ作成を開始します...")
success_individual = 0
error_count = 0

for dirpath, files_in_dir in folder_to_files.items():
    last_folder_name = os.path.basename(dirpath)
    
    for col_name in target_columns:
        plt.figure(figsize=(10, 5))
        valid_files_count = 0
        
        for file_path in files_in_dir:
            try:
                df = pd.read_csv(file_path, header=2)
                df = df.drop(0).reset_index(drop=True)
                if col_name not in df.columns: continue
                    
                df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
                df_clean = df.dropna(subset=[col_name])
                N = len(df_clean)
                if N == 0: continue

                y = df_clean[col_name].values
                F = np.fft.fft(y)
                amp = np.abs(F) / (N / 2)
                amp[0] /= 2
                freq = np.fft.fftfreq(N, d=dt)
                
                mask = (freq >= 0)
                plt.plot(freq[mask], amp[mask], alpha=0.5, linewidth=1.0)
                valid_files_count += 1
                
            except Exception as e:
                error_count += 1
                
        if valid_files_count > 0:
            plt.xlabel('Frequency [Hz]')
            plt.ylabel('Amplitude [MPa]')
            plt.title(f'FFT Spectrum - {last_folder_name} [{col_name}] (Overlaid {valid_files_count} files)')
            plt.xlim(0, 500)
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            
            output_image_name = f"{last_folder_name}_{col_name}_combined_fft.png"
            plt.savefig(os.path.join(output_dir_path, output_image_name), dpi=150)
            success_individual += 1
            print(f"  --> ✅ 保存: {output_image_name}")
            
        plt.close()

print("-" * 30)
print(f"🏁 個別グラフ完了！（生成: {success_individual}枚, エラー: {error_count}件）")
print("👉 次の「セル6」を実行してください。")

In [ ]:
# セル6：全列×全ファイルの統合グラフ作成（統合版）

print("🚀 【統合版】全指定列のグラフ作成を開始します...")
success_integrated = 0

# (念のためセル4で定義されているかチェック)
if 'output_dir_path' not in locals():
    print("⚠️ 警告: セル4が実行されていない可能性があります。")

for dirpath, files_in_dir in folder_to_files.items():
    last_folder_name = os.path.basename(dirpath)
    
    plt.figure(figsize=(12, 6))
    valid_integrated_count = 0
    plotted_labels = set()
    
    for file_path in files_in_dir:
        try:
            df = pd.read_csv(file_path, header=2)
            df = df.drop(0).reset_index(drop=True)
            file_plotted = False
            
            for i, col_name in enumerate(target_columns):
                if col_name not in df.columns: continue
                
                df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
                df_clean = df.dropna(subset=[col_name])
                N = len(df_clean)
                if N == 0: continue

                y = df_clean[col_name].values
                F = np.fft.fft(y)
                amp = np.abs(F) / (N / 2)
                amp[0] /= 2
                freq = np.fft.fftfreq(N, d=dt)
                mask = (freq >= 0)
                
                # 凡例の重複防止
                label_name = col_name if col_name not in plotted_labels else ""
                
                plt.plot(freq[mask], amp[mask], color=colors[i % len(colors)], 
                         alpha=0.4, linewidth=1.0, label=label_name)
                
                if label_name:
                    plotted_labels.add(col_name)
                    
                file_plotted = True
                
            if file_plotted:
                valid_integrated_count += 1
                
        except Exception:
            pass # エラーはセル5でカウント済みのためスキップ
            
    if valid_integrated_count > 0:
        plt.xlabel('Frequency [Hz]')
        plt.ylabel('Amplitude [MPa]')
        plt.title(f'FFT Spectrum - {last_folder_name} [ALL COLUMNS] (Overlaid {valid_integrated_count} files)')
        plt.xlim(0, 500)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend(loc='upper right')
        plt.tight_layout()
        
        # 1. フォルダへの画像保存
        output_image_name_all = f"{last_folder_name}_ALL_combined_fft.png"
        plt.savefig(os.path.join(output_dir_path, output_image_name_all), dpi=150)
        success_integrated += 1
        print(f"  --> 🌟 保存: {output_image_name_all}")
        
        # 2. Jupyter Notebookの画面上に表示
        plt.show()
        
    else:
        # 描画するものがなかった場合はメモリ解放だけ行う
        plt.close()

print("-" * 30)
print(f"🏁 統合グラフ完了！（生成: {success_integrated}枚）")
print("🎉 すべての処理が正常に終了しました！出力フォルダを確認してください。")